# Part 2, Topic 2: Voltage Glitching to Bypass a Password Check (Hands-on Explorer)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *We've seen how voltage glitching can corrupt a simple counting loop. Now let's see whether the same technique can make a firmware password check accept a wrong answer - and whether "stronger" settings or the specific guess you send change anything.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware, and seeing why they do not simply carry over
* Searching a wider parameter space when a narrow one does not turn anything up
* Checking whether a fault depends on the data sent, or only on timing


---
## Hands-on edition: how to use this notebook

This notebook is the original lab plus **seven experiments** where *you* change the numbers and watch what a voltage glitch does to a real password check. Run a cell, change a number, run it again.

| You see | It means |
|---|---|
| a code cell that starts with **`>>> EDIT ME <<<`** | Change the numbers between the two `>>>` lines, then run the cell (**Shift+Enter**). |
| a code cell that starts with **`RUN ONLY (do not edit)`** | Helper code. Run it once (top to bottom) and leave it alone. |
| a short markdown cell titled **My prediction** | Write down what you *expect* before you run the cell, and what you *saw* afterwards. |

**Before you start (hardware checklist)** - same kit as Fault 2_1:

1. ChipWhisperer-**Husky** plugged straight into a USB port of the computer (not through a hub).
2. **CW312T-SAM4S** target seated on the **CW313** board, connected with the ribbon cable.
3. The Husky **`Crowbar`** SMA cable connected to the CW313's second SMA socket.
4. No other notebook using the Husky (**File > Shut Down Notebook** in any old notebook).

**A warning before you start:** in Fault 2_1, settings that worked sat in one narrow, predictable band. In this lab the useful settings turned out to be in a completely different part of the search space than the one that merely crashes the target - Experiments 1 to 3 walk into that surprise on purpose, and Experiment 4 is how we found our way out of it. Do not skip ahead expecting Fault 2_1's numbers to work here.

## The firmware being tested

`password()` in `firmware/mcu/simpleserial-glitch/simpleserial-glitch.c` is a small test routine built for this lab:

```c
char passwd[] = "touch";
char passok = 1;
trigger_high();
for(cnt = 0; cnt < 5; cnt++){
    if (pw[cnt] != passwd[cnt]){
        passok = 0;
    }
}
trigger_low();
simpleserial_put('r', 1, (uint8_t*)&passok);
```

`"touch"` is a constant baked into this open-source demo firmware, not a secret anyone is hiding - knowing it does not help you here. The point of the lab is the *shape* of the check: `passok` starts at 1 and is only ever set to 0; the loop always runs all 5 comparisons (unlike the SCA101 password lab, it never stops early, so there is no timing side-channel to measure). The only way to make a **wrong** guess come back as PASS is to corrupt the program *while it runs* - skip the `passok = 0;` assignment, corrupt the loop variable, or otherwise disturb execution during the comparison - which is exactly what a voltage glitch tries to do.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()

if scope._is_husky is False:
    if PLATFORM == "CWLITEXMEGA":
        scope.clock.clkgen_freq = 32E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*32/7.37
        else:
            target.baud = 38400*32/7.37
    elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
        scope.clock.clkgen_freq = 24E6
        if SS_VER=='SS_VER_2_1':
            target.baud = 230400*24/7.37
        else:
            target.baud = 38400*24/7.37
        time.sleep(0.1)

### Sanity check: does the password check work at all, with no glitch?

**Do this:** run the next cell as it is. It sends the correct password, then two wrong ones, with no glitching.

In [ ]:
reboot_flush()
CORRECT_PW = b"touch"

def check(guess):
    target.flush()
    target.simpleserial_write('p', guess)
    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)
    passok = val['payload'][0] if val['valid'] else None
    print("guess={!r:10} passok={}".format(bytes(guess), passok))

check(CORRECT_PW)
check(b"xxxxx")
check(b"aaaaa")

**You should see:** `passok=1` for `touch`, `passok=0` for the two wrong guesses. If you see anything else, fix that before going any further - every experiment below depends on this baseline being correct.

## Part 1: does Fault 2_1's glitch work here too?

Like with clock glitching, the scope object can set typical glitch settings for you - with the extra choice of which transistor to use (`'hp'`, `'lp'`, or `'both'`):

In [ ]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False)  # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False)  # use both transistors

In [ ]:
# ------------------------------------  RUN ONLY (do not edit)  ------------------------------------
# Helper functions for the experiments. Run this cell once; do not edit it.
import time, random
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

cw.set_all_log_levels(cw.logging.CRITICAL)
scope.adc.timeout = 0.3
if scope._is_husky:
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True

OUTCOME_COLORS = {"normal": "#c9d1d9", "reset": "#d9534f", "success": "#2e9e4f"}


def set_glitch(width, offset, ext_offset, repeat=1):
    if scope._is_husky:
        width, offset = int(round(width)), int(round(offset))
    scope.glitch.width = width
    scope.glitch.offset = offset
    scope.glitch.ext_offset = int(ext_offset)
    scope.glitch.repeat = int(repeat)


def fire_glitch(guess, width, offset, ext_offset, repeat=1):
    """Fire ONE glitch while the target checks `guess`. Returns ("normal"|"reset"|"success", passok or None).
    'success' means the check said PASS even though `guess` does not match the firmware's password."""
    set_glitch(width, offset, ext_offset, repeat)
    target.flush()
    if scope.adc.state:
        reboot_flush()
    scope.arm()
    target.simpleserial_write('p', guess)
    timed_out = scope.capture()
    scope.io.vglitch_reset()
    if timed_out:
        reboot_flush()
        return "reset", None
    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)
    if val['valid'] is False:
        reboot_flush()
        return "reset", None
    passok = val['payload'][0]
    outcome = "success" if (passok == 1 and guess != CORRECT_PW) else "normal"
    return outcome, passok


def fire_many(guess, width, offset, ext_offset, repeat=1, shots=20):
    tally = {"normal": 0, "reset": 0, "success": 0}
    cw.set_all_log_levels(cw.logging.CRITICAL)
    for _ in range(shots):
        outcome, _ = fire_glitch(guess, width, offset, ext_offset, repeat)
        tally[outcome] += 1
    return tally


def scan(guess, points, shots):
    tallies = []
    for p in points:
        tallies.append(fire_many(guess, *p, shots=shots))
        print(".", end="", flush=True)
    print()
    return tallies


def plot_tallies(labels, tallies, xlabel, title):
    fig, ax = plt.subplots(figsize=(9, 3.4))
    bottom = np.zeros(len(labels))
    for name in ("success", "reset", "normal"):
        share = np.array([100.0 * t[name] / max(1, sum(t.values())) for t in tallies])
        ax.bar(range(len(labels)), share, bottom=bottom, color=OUTCOME_COLORS[name], label=name, width=0.85)
        bottom += share
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([str(l) for l in labels], rotation=90 if len(labels) > 12 else 0, fontsize=8)
    ax.set_xlabel(xlabel); ax.set_ylabel("% of shots"); ax.set_title(title); ax.set_ylim(0, 100)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


def print_table(labels, tallies, label_name):
    print("{:>10} {:>8} {:>8} {:>8}".format(label_name, "normal", "reset", "success"))
    for l, t in zip(labels, tallies):
        print("{:>10} {:>8} {:>8} {:>8}".format(str(l), t["normal"], t["reset"], t["success"]))


def glitch_on(hp=True, lp=False):
    scope.io.glitch_hp = hp
    scope.io.glitch_lp = lp


def recover():
    """Run this if you stopped a cell half way: reboots the target, puts the glitch hardware back."""
    scope.io.vglitch_reset()
    reboot_flush()
    glitch_on(True, False)
    scope.glitch.repeat = 1
    print("target rebooted, glitch hardware restored (hp on, lp off, repeat 1)")


print("helpers ready: fire_glitch, fire_many, scan, plot_tallies, print_table, glitch_on, recover")

### Experiment 1: try Fault 2_1's winning setting on this firmware

**Goal:** find out whether a glitch setting that worked for the counting loop (width 1770, offset 2670, ext_offset 1) does anything useful here too.

**Do this:** run the cell as it is, with a **wrong** guess. Then try the neighbours below.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS      = b"xxxxx"  # a WRONG guess (5 bytes; anything that is not CORRECT_PW)
WIDTH      = 1770       # Fault 2_1's winning width
OFFSET     = 2670       # Fault 2_1's winning offset
EXT_OFFSET = 1           # Fault 2_1's winning clock cycle
REPEAT     = 1
SHOTS      = 20
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

t0 = time.time()
tally = fire_many(GUESS, WIDTH, OFFSET, EXT_OFFSET, REPEAT, SHOTS)
print("guess={!r} width={} offset={} ext_offset={}   |   {} shots in {:.1f} s".format(
    bytes(GUESS), WIDTH, OFFSET, EXT_OFFSET, SHOTS, time.time() - t0))
for name in ("normal", "reset", "success"):
    print("  {:8s} {:3d}  {}".format(name, tally[name], "#" * tally[name]))

**What you should see on our kit:** almost every shot is `reset` - the counting-loop setting just **crashes** the password check; it never flips `passok`. This is the central lesson of this lab: a glitch setting is a property of *one piece of code running at one moment*, not of the chip in general. The same width/offset that calmly disturbed a 2500-count loop is far too strong for whatever is happening here at that instant. Try a few wrong guesses (`b"aaaaa"`, `b"zzzzz"`) - the behaviour does not depend on which wrong guess you send.

**My prediction (Experiment 1):** _double-click and write here_

**What I saw:** _..._

### Experiment 2: map the crash edge for this firmware

**Goal:** Experiment 1's setting was deep inside a crash region. Let's find the **edge** of that region for this particular code, the same way Fault 2_1 did for the counting loop.

**Do this:**

1. Run the cell as it is: it scans `WIDTH_START` to `WIDTH_STOP` at a fixed offset and ext_offset, with a wrong guess.
2. Change `EXT_OFFSET` to 2, then to 0. Does the crash region change?
3. Zoom in with `WIDTH_STEP = 2` around the edge you find.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS       = b"xxxxx"
OFFSET      = 2670
EXT_OFFSET  = 1
WIDTH_START = 1700
WIDTH_STOP  = 1800
WIDTH_STEP  = 5
SHOTS       = 10
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

widths = list(range(WIDTH_START, WIDTH_STOP + 1, WIDTH_STEP))
t0 = time.time()
tallies = scan(GUESS, [(OFFSET, w, EXT_OFFSET) if False else (w, OFFSET, EXT_OFFSET) for w in widths], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(widths, tallies, "WIDTH", "offset={}  ext_offset={}".format(OFFSET, EXT_OFFSET))
print_table(widths, tallies, "width")

**What you should see on our kit:** at `ext_offset=1`, width 1750 and below is `normal`; a transition region around 1758-1764 is a mix of `normal` and `reset`; width 1766 and above is `reset` every time. `ext_offset=2` gives a very similar edge a couple of steps earlier. `ext_offset=0` (and every value from 3 up to about 5) does not disturb the target at all across this whole width range - only cycles 1 and 2 are sensitive here, exactly like Fault 2_1's counting loop. **No width in this scan ever produced a `success`** - crashing the check is easy; making it lie is not, at least not from this spot.

**My prediction (Experiment 2):** _double-click and write here_

**What I saw (where is the edge? does `success` ever appear?):** _..._

### Experiment 3: scan `ext_offset` widely - is there a bypass hiding later in the trigger window?

**Goal:** Fault 2_1's vulnerable window was only 2-3 clock cycles wide. This password check's trigger window stays high for much longer (it covers five comparisons plus the serial write back). Let's scan a wide range of `ext_offset` to see what the rest of that window looks like.

**Do this:** run the cell as it is (a width just below Experiment 2's edge, scanned across `ext_offset` 0 to 60). Watch for any `success` bars, and note roughly how often `reset` appears.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS      = b"xxxxx"
WIDTH      = 1762
OFFSET     = 2670
EXT_START  = 0
EXT_STOP   = 60
EXT_STEP   = 1
SHOTS      = 8
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

exts = list(range(EXT_START, EXT_STOP + 1, EXT_STEP))
t0 = time.time()
tallies = scan(GUESS, [(WIDTH, OFFSET, e) for e in exts], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(exts, tallies, "EXT_OFFSET", "width={}  offset={}".format(WIDTH, OFFSET))
any_success = [e for e, t in zip(exts, tallies) if t["success"]]
print("ext_offset values with at least one success:", any_success)

**What you should see on our kit:** `reset` shows up in short bursts every 20-ish cycles (around 1-2, 20-25, 33, 40-42, 53 and 59-60) with `normal` everywhere else - a periodic pattern, probably the serial/UART handling inside the trigger window rather than the password comparison itself. **Still no `success` anywhere in this range, at this width and offset.** We have now swept a width that sits right at an edge, across 60 different moments in time, and found nothing: the useful fault setting, if one exists, is not a small tweak away from where we have been looking.

**My prediction (Experiment 3):** _double-click and write here_

**What I saw:** _..._

## Part 2: widen the net

### Experiment 4: a randomized search over a much bigger space

**Goal:** three experiments in the 'obvious' area (small width, offset around 2670, ext_offset under 60) found nothing but crashes. Rather than inching the ranges wider by hand, let the computer pick *random* combinations across a much bigger box and watch for the first `success`.

**Do this:** run the cell as it is. **This can take a few minutes** - it keeps going until it finds one success or runs out of time. Watch the progress line.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS           = b"xxxxx"
WIDTH_RANGE     = (1500, 2100)   # Husky: 0 to 2296
OFFSET_RANGE    = (1800, 3300)   # Husky: 0 to 4592
EXT_OFFSET_RANGE = (0, 150)
TIME_BUDGET_S   = 240            # give up after this many seconds
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

random.seed()                      # different random settings every time you run this
cw.set_all_log_levels(cw.logging.CRITICAL)
t0 = time.time()
attempts = 0
tally = {"normal": 0, "reset": 0, "success": 0}
found = None
while time.time() - t0 < TIME_BUDGET_S:
    w = random.randint(*WIDTH_RANGE)
    o = random.randint(*OFFSET_RANGE)
    e = random.randint(*EXT_OFFSET_RANGE)
    outcome, _ = fire_glitch(GUESS, w, o, e)
    tally[outcome] += 1
    attempts += 1
    if outcome == "success":
        found = (w, o, e)
        print("\nFOUND a bypass: width={} offset={} ext_offset={}  ({} attempts, {:.0f} s)".format(
            w, o, e, attempts, time.time() - t0))
        break
    if attempts % 200 == 0:
        print(".", end="", flush=True)
cw.set_all_log_levels(cw.logging.WARNING)
print("\n{} attempts in {:.0f} s -> {}".format(attempts, time.time() - t0, tally))
if found is None:
    print("No bypass found in the time budget - that is a valid result too. Run it again (more randomness), "
          "or raise TIME_BUDGET_S.")

**What you should see on our kit:** this is the one experiment in this notebook where luck genuinely matters, and how *much* luck varies between runs. In one run, a search like this one tried **7248 settings over about 230 seconds and found only 2 successes** (roughly 1 in 3600 attempts). Run again later in the same session, it found one in **1149 attempts, about 46 seconds** - a very different hit rate, which fits the drift you will see documented after Experiment 5: the size of the 'good' region itself seems to change over time, not just its exact location. What stayed consistent across every run: successes shared a **much bigger width (close to 1970-1990) and a much bigger offset (close to 3000-3100)** than anything in Experiments 1 to 3 - a completely different corner of the search space from the one that merely crashes the target. If your own run does not find one in the time budget, that is a real and useful outcome: it means **this fault, from this glitch circuit, did not defeat this check within the search you tried** - not every combination of fault and target gives in, and knowing that a given approach does *not* work is part of an honest security evaluation too.

**My prediction (Experiment 4):** _double-click and write here_

**What I saw (did you find one? how many attempts did it take?):** _..._

### Experiment 5: now that you have a lead, zoom in

**Goal:** turn one lucky random hit into a setting you can rely on, and see how wide the 'good' neighbourhood actually is.

**Do this:**

1. Put the `(width, offset, ext_offset)` Experiment 4 found into `CENTER` below (or use our kit's as a starting point if yours did not find one yet).
2. Run the cell: it tries a small grid of neighbours around `CENTER`, all at the same `ext_offset`, and reports the success count out of `SHOTS` at each point.
3. Change `EXT_OFFSET_TRY` to a few cycles either side of `CENTER`'s. Is the neighbourhood just as wide there?

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS          = b"xxxxx"
CENTER         = (1979, 3071, 121)   # (width, offset, ext_offset) - a success we found on our kit
WIDTH_STEPS    = (-10, -5, 0, 5, 10)
OFFSET_STEPS   = (-20, -10, 0, 10, 20)
SHOTS          = 8
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

cw0, co0, ce0 = CENTER
t0 = time.time()
print("{:>8} {:>8} {:>9} {:>8} {:>8}".format("width", "offset", "success", "reset", "normal"))
for dw in WIDTH_STEPS:
    for do in OFFSET_STEPS:
        w, o = cw0 + dw, co0 + do
        t = fire_many(GUESS, w, o, ce0, shots=SHOTS)
        marker = " <--" if t["success"] else ""
        print("{:>8} {:>8} {:>9} {:>8} {:>8}{}".format(w, o, t["success"], t["reset"], t["normal"], marker))
print("done in {:.0f} s".format(time.time() - t0))

**What you should see on our kit:** success is **not** a single pinpoint - most of a 20-by-40-step block around `(1979, 3071)` gave at least one success out of 8 shots when we first mapped it, with individual points ranging from about 10% to 40%. `ext_offset` was pickier: only a few specific values (121 and 130 among the ones we tried, and 89 at a different width/offset) gave any success at all there - most neighbouring ext_offsets gave nothing.

**GOOD TO KNOW - these settings drift.** When we came back to the *exact same* `(1979, 3071, 121)` later in the same session, it gave **zero** successes in 60 shots - not one. A nearby point our random search found independently, `(1973, 3083, 141)`, was still giving occasional successes at that point. Nothing about the wiring or the code had changed; the glitch circuit's exact sweet spot moves over time (likely with the board's temperature and small clock drift), by more than you might expect. **Treat every specific number in this notebook as 'this was possible,' not as a guarantee you can reproduce on demand** - always re-measure with fresh shots before trusting a setting, and if one stops working, try its near neighbours before assuming it never worked at all.

**My prediction (Experiment 5):** _double-click and write here_

**What I saw (how wide is the good neighbourhood in width/offset? in ext_offset?):** _..._

### Experiment 6: does a *stronger* glitch work better here too?

**Goal:** Fault 2_1's Experiment 6 found that `hp + lp` together crashed the target at a smaller width than `hp` alone - i.e. a stronger glitch. Does using both transistors also help bypass *this* check?

**Do this:** run the cell at your best setting from Experiment 5, once with `hp` only and once with `hp + lp`.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
GUESS   = b"xxxxx"
WIDTH, OFFSET, EXT_OFFSET = 1979, 3071, 121   # your best setting from Experiment 5
SHOTS   = 40
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

scope.vglitch_setup('hp', default_setup=False)
t_hp = fire_many(GUESS, WIDTH, OFFSET, EXT_OFFSET, shots=SHOTS)
print("hp only   ", t_hp, " success rate = {:.0f}%".format(100 * t_hp["success"] / SHOTS))

scope.vglitch_setup('both', default_setup=False)
t_both = fire_many(GUESS, WIDTH, OFFSET, EXT_OFFSET, shots=SHOTS)
print("hp + lp   ", t_both, " success rate = {:.0f}%".format(100 * t_both["success"] / SHOTS))

scope.vglitch_setup('hp', default_setup=False)   # put it back for the later experiments

**What you should see on our kit - and it is a surprise, with a caveat:** in one back-to-back comparison at a setting that was working at the time, `hp` alone gave about 15% success while `hp + lp` together gave **0%** - every shot was a `reset`. The stronger glitch did not improve anything there; it pushed the disturbance past the useful instant straight into a crash, the opposite of Fault 2_1's counting loop, where `hp + lp` was the stronger, better choice. **The caveat:** because these settings drift (see the note after Experiment 5), a single comparison like this is a data point, not a law - run it again at your own current best setting, with fresh shots for both `hp` and `hp + lp`, before you trust which one wins. What should **not** change is the general lesson: which transistor combination helps depends on the code being glitched, not just on the physical circuit, so it is always worth comparing both rather than assuming 'stronger is always better'.

**My prediction (Experiment 6):** _double-click and write here_

**What I saw:** _..._

### Experiment 7: how reliable is it, and does the guess itself matter?

**Goal:** measure your best setting's success rate properly, and check something important: does the *content* of the wrong guess change anything? (In the SCA101 power-analysis lab, the guess mattered enormously - that attack is a side channel. This fault either works or it doesn't, regardless of what you send - if that is true, it tells you this is a pure control-flow corruption, not a leak of any kind.)

**Do this:**

1. Run the cell as it is: it fires `SHOTS` glitches at your best setting for each guess in `GUESSES`.
2. Add a guess of your own (any 5 bytes) to the list.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH, OFFSET, EXT_OFFSET = 1979, 3071, 121
SHOTS   = 20
GUESSES = [b"xxxxx", b"aaaaa", b"zzzzz", b"\x00\x00\x00\x00\x00", b"touc0"]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

scope.vglitch_setup('hp', default_setup=False)
t0 = time.time()
print("{:>18} {:>9} {:>8} {:>8}".format("guess", "success", "reset", "normal"))
for g in GUESSES:
    t = fire_many(g, WIDTH, OFFSET, EXT_OFFSET, shots=SHOTS)
    print("{:>18} {:>9} {:>8} {:>8}  ({:.0f}%)".format(repr(bytes(g)), t["success"], t["reset"], t["normal"],
                                                           100 * t["success"] / SHOTS))
print("done in {:.0f} s".format(time.time() - t0))

**What you should see on our kit:** run all five guesses back-to-back (so drift has the least chance to matter between them) and compare their success counts *to each other*, not to a fixed target percentage - on our kit they came out close to each other (none was dramatically easier or harder), whatever the absolute level happened to be that moment. That confirms the lesson from the firmware listing: the fault corrupts the comparison's *outcome*, not a part of it that depends on which byte was wrong, so there is nothing to be gained by choosing a 'smarter' wrong guess - unlike a real side-channel attack, where the guess is everything.

**My prediction (Experiment 7):** _double-click and write here_

**What I saw (does the guess matter? what is the success rate?):** _..._

## Challenges (optional)

1. **Characterize the second spot.** Our random search also found a success near `(1985, 2999, 89)`. Map its neighbourhood the way Experiment 5 mapped `(1979, 3071, 121)`. Is it as wide?
2. **A tighter automatic search.** Now that you know roughly where the good region is, write a `GlitchController` search (like Fault 2_1 Experiment 4) with `width` and `offset` ranges centred on it. How much faster is it than the blind random search in Experiment 4?
3. **Does `repeat` help or hurt?** Try `REPEAT = 2` at your best setting. Does the success rate go up or down?
4. **A fourth spot.** Run Experiment 4 again (a new random seed) for several minutes. Does it always find settings with a large width and a large offset, or can a bypass exist somewhere else too?

## If something goes wrong

| What you see | Most likely cause | What to do |
|---|---|---|
| The sanity check does not print `passok=1` for `touch` | Firmware did not flash, or stale serial buffer | Re-run the build/flash cells; re-run `reboot_flush()` |
| Every shot is `normal`, never `reset` or `success`, even at a huge width | The Crowbar SMA cable is not connected | Fit the cable (see Fault 2_1's guide), run the setup cells again |
| Every shot is `reset` | You are inside a crash region for this width/offset | Try a smaller width, or a different part of the search space (Experiment 4) |
| Experiment 4 finds nothing in the time budget | This is a real result, not a bug - see Experiment 4's 'what you should see' | Run it again (new random settings) or raise `TIME_BUDGET_S` |
| `OSError: Could not find ChipWhisperer` | Husky unplugged, or another notebook still holds it | Plug it in; shut down every other notebook; restart this kernel |
| `USBErrorPipe` / `Unknown hwInfoVer` | The Husky's USB link or FPGA stopped answering | Unplug and replug the Husky directly into the computer; restart the kernel |
| A cell was interrupted (Stop button) | Target or glitch hardware may be left in an odd state | Run `recover()` in a new cell |


## Finish: switch the glitch hardware off

The last cell switches the glitch transistors off and disconnects the Husky and the target. **Always run it before closing the notebook.**

In [ ]:
scope.io.glitch_hp = False
scope.io.glitch_lp = False
scope.glitch.enabled = False
scope.dis()
target.dis()